## **In-Context Learning (Few-Shot Prompting)**

Instead of traditional fine-tuning or training a separate classification layer, we will leverages the pre-trained models capabilities of Large Language Models (LLMs)—specifically Llama-3.2-3B From Facebook, Gemma from Google, and Phi-3—to From Microsoft to Classify the ATIS Intent.

Key components of this specific implementation include:

1. **Prompt Engineering**: A structured template is used to guide the model by listing all possible intent categories and providing specific "few-shot" examples.

2. **Unsloth Optimization**: The notebook utilizes the Unsloth library to perform efficient inference. This includes 4-bit quantization via load_in_4bit=True, which allows these large models to run on standard hardware like a free Google Colab GPU.

3. **FastLanguageModel**: The FastLanguageModel from the Unsloth library is employed to load the models and prepare them specifically for high-speed inference.

this code implements In-Context Learning. Instead of training a specific classifier from scratch, We are "programming" a pre-trained general-purpose AI Llama-3.2-3B From Facebook, Gemma from Google, and Phi-3—to From Microsof using natural language instructions to perform a specific task (ATIS Intent Classification) efficiently.

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU est disponible.")
    print(f"Nom du GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU n'est pas disponible.")

GPU est disponible.
Nom du GPU: Tesla T4


In [ ]:
!pip install unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225

In [ ]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:

ATIS_INTENT_MAPPING = {
    'abbreviation': "Abbreviation and Fare Code Meaning Inquiry",
    'aircraft': "Aircraft Type Inquiry",
    'airfare': "Airfare and Fares Questions",
    'airline': "Airline Information Request",
    'airport': "Airport Information and Queries",
    'capacity': "Aircraft Seating Capacity Inquiry",
    'cheapest': "Cheapest Fare Inquiry",
    'city': "Airport Location Inquiry",
    'distance': "Airport Distance Inquiry",
    'flight': "Flight Booking Request",
    'flight_no': "Flight Number Inquiry",
    'flight_time': "Time Inquiry",
    'ground_fare': "Ground Transportation Cost Inquiry",
    'ground_service': "Ground Transportation Inquiry",
    'ground_service+ground_fare': "Airport Ground Transportation and Cost Query",
    'meal': "Inquiry about In-flight Meals",
    'quantity': "Flight Quantity Inquiry",
    'restriction': "Flight Restriction Inquiry"
}

### **Llama**

In [ ]:
# Model & Tokenizer Initialization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=256,
    load_in_4bit=True,  # fits in free Colab GPU
)

# Setting up for Inference
# This command puts the model in a "read-only" optimized mode.
# It disables features needed only for training (like dropout and gradients), which speeds up the response time and reduces memory consumption further.
FastLanguageModel.for_inference(model)

LABELS = list(ATIS_INTENT_MAPPING.values())

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [ ]:
# The Classification Function (classify_intent_llama)

# This function follows the Prompt Engineering pattern:

# Prompt Construction: * It takes all the possible categories (the LABELS) and lists them clearly.

# It provides the specific user query (text).

# It sets a strict constraint: "Reply with only the intent name, nothing else." This prevents the LLM from being "chatty" and giving you full sentences when you only need a label.

def classify_intent_llama(text):
    label_list = "\n".join(f"- {l}" for l in LABELS)
    prompt = f"""Classify the following airline query into EXACTLY ONE of these intents:
{label_list}

Query: "{text}"
Reply with only the intent name, nothing else.
Intent:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=20, temperature=1, do_sample=False)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split("Intent:")[-1].strip()

In [ ]:
prediction = classify_intent_llama("i need flights that arrive in baltimore from ankara")

Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print(prediction)

Flight Number Inquiry


In [ ]:
predictions = [
    classify_intent_llama("i need flights that arrive in baltimore from ankara"),
    classify_intent_llama("what is the cheapest fare from new york to chicago"),
    classify_intent_llama("which airlines fly from boston to denver")
    ]

Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
predictions

['Flight Number Inquiry',
 'Cheapest Fare Inquiry\nIntent',
 'Airline Information Request']

### **Gemma**

In [ ]:
gemma_model, gemma_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-2b-it",
    max_seq_length=256,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(gemma_model)

==((====))==  Unsloth 2026.4.6: Fast Gemma2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.


Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
          (rotary_emb): GemmaFixedRotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_atte

In [ ]:
def classify_intent_gemma(text):
    label_list = "\n".join(f"- {l}" for l in LABELS)
    prompt = f"""<start_of_turn>user
Classify the following airline query into EXACTLY ONE of these intents:
{label_list}

Query: "{text}"
Reply with only the intent name, nothing else.<end_of_turn>
<start_of_turn>model
"""

    inputs = gemma_tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = gemma_model.generate(
        **inputs,
        max_new_tokens=20,
        temperature=1,
        do_sample=False,
    )
    result = gemma_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Gemma appends its reply after the last <start_of_turn>model token
    return result.split("<start_of_turn>model")[-1].strip()

In [ ]:
gemma_prediction = classify_intent_gemma("i need flights that arrive in baltimore from ankara")
print(prediction)

Both `max_new_tokens` (=20) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Flight Number Inquiry


In [ ]:
gemma_predictions = [
    classify_intent_gemma("i need flights that arrive in baltimore from ankara"),
    classify_intent_gemma("what is the cheapest fare from new york to chicago"),
    classify_intent_gemma("which airlines fly from boston to denver")
    ]

Both `max_new_tokens` (=20) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
gemma_predictions

['Flight Number Inquiry',
 'Cheapest Fare Inquiry\nIntent',
 'Airline Information Request']

### **Phi-3**

In [ ]:
phi3_model, phi3_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct",
    max_seq_length=256,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(phi3_model)

==((====))==  Unsloth 2026.4.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((3072,), eps=1e-05)
        (post_attention_laye

In [ ]:
def classify_intent_phi3(text):
    label_list = "\n".join(f"- {l}" for l in LABELS)
    prompt = f"""<|user|>
Classify the following airline query into EXACTLY ONE of these intents:
{label_list}

Query: "{text}"
Reply with only the intent name, nothing else.<|end|>
<|assistant|>
"""

    inputs = phi3_tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = phi3_model.generate(
        **inputs,
        max_new_tokens=20,
        temperature=1,
        do_sample=False,
    )
    result = phi3_tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Phi-3 appends its reply after the last <|assistant|> token
    return result.split("<|assistant|>")[-1].strip()

In [ ]:
phi3_prediction = classify_intent_phi3("i need flights that arrive in baltimore from ankara")

Both `max_new_tokens` (=20) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print(phi3_prediction)

Classify the following airline query into EXACTLY ONE of these intents:
- Abbreviation and Fare Code Meaning Inquiry
- Aircraft Type Inquiry
- Airfare and Fares Questions
- Airline Information Request
- Airport Information and Queries
- Aircraft Seating Capacity Inquiry
- Cheapest Fare Inquiry
- Airport Location Inquiry
- Airport Distance Inquiry
- Flight Booking Request
- Flight Number Inquiry
- Time Inquiry
- Ground Transportation Cost Inquiry
- Ground Transportation Inquiry
- Airport Ground Transportation and Cost Query
- Inquiry about In-flight Meals
- Flight Quantity Inquiry
- Flight Restriction Inquiry

Query: "i need flights that arrive in baltimore from ankara"
Reply with only the intent name, nothing else. Flight Booking Request


In [ ]:
phi3_predictions = [
    classify_intent_phi3("i need flights that arrive in baltimore from ankara"),
    classify_intent_phi3("what is the cheapest fare from new york to chicago"),
    classify_intent_phi3("which airlines fly from boston to denver")
    ]

Both `max_new_tokens` (=20) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
phi3_predictions

['Classify the following airline query into EXACTLY ONE of these intents:\n- Abbreviation and Fare Code Meaning Inquiry\n- Aircraft Type Inquiry\n- Airfare and Fares Questions\n- Airline Information Request\n- Airport Information and Queries\n- Aircraft Seating Capacity Inquiry\n- Cheapest Fare Inquiry\n- Airport Location Inquiry\n- Airport Distance Inquiry\n- Flight Booking Request\n- Flight Number Inquiry\n- Time Inquiry\n- Ground Transportation Cost Inquiry\n- Ground Transportation Inquiry\n- Airport Ground Transportation and Cost Query\n- Inquiry about In-flight Meals\n- Flight Quantity Inquiry\n- Flight Restriction Inquiry\n\nQuery: "i need flights that arrive in baltimore from ankara"\nReply with only the intent name, nothing else. Flight Booking Request',
 'Classify the following airline query into EXACTLY ONE of these intents:\n- Abbreviation and Fare Code Meaning Inquiry\n- Aircraft Type Inquiry\n- Airfare and Fares Questions\n- Airline Information Request\n- Airport Informati

Loading ATIS dataset...
Train: 4209 | Valid: 743 | Test: 876
Number of intent categories: 17

MODEL 1: TF-IDF + Logistic Regression
Training Time:   0.43 seconds
Inference Time:  0.0029 seconds (for 876 samples)
Accuracy:        0.8984
Macro-F1 Score:  0.5033

Detailed Report:
                precision    recall  f1-score   support

  abbreviation       1.00      0.97      0.98        33
      aircraft       0.47      0.89      0.62         9
       airfare       0.91      0.85      0.88        48
       airline       1.00      0.74      0.85        38
       airport       1.00      0.11      0.20        18
      capacity       1.00      0.33      0.50        21
      cheapest       0.00      0.00      0.00         0
          city       0.00      0.00      0.00         6
      distance       1.00      0.30      0.46        10
        flight       0.90      0.99      0.94       632
     flight_no       0.00      0.00      0.00         8
   flight_time       0.00      0.00      0.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

NameError: name 'FastLanguageModel' is not defined